# Cloud AI — Colab backend on Unsloth

Runs the **Cloud AI router** (`backend/server.py`) with `CLOUD_BACKEND=unsloth`: the model is loaded in-process by [Unsloth](https://unsloth.ai) and served through a Cloudflare quick tunnel. Use it to serve a base model, or a LoRA adapter you fine-tuned with Unsloth, without exporting to GGUF.

1. *Runtime → Change runtime type → GPU* (T4 is enough for a 4-bit 4B model).
2. *Runtime → Run all*. Approve the Drive prompt if `MODEL` or `MOUNT_DRIVE` needs Drive.
3. The last cell prints `CLOUD_URL` and `CLOUD_TOKEN`. In VS Code open *Settings → Cloud AI* and paste them into **Tunnel Url** and **Token**.

Driving this from a terminal instead: `backend/colab_up.sh` does the same with the Colab CLI, including the Drive mount.

Image turns need Ollama as well (see `cloud_colab.ipynb`); this notebook serves text and tool calling only.

In [ ]:
#@title 1 · Configuration
MODEL       = "unsloth/Qwen3-4B-Instruct-2507"  #@param {type:"string"}
#          or a Drive folder with a saved adapter, e.g. /content/drive/MyDrive/CASMI26/runs/qwen3-4b-v1/lora
TEMPLATE    = ""       #@param {type:"string"}
MOUNT_DRIVE = "auto"   #@param ["auto", "yes", "no"]
REPO        = "https://github.com/cloud-fronts-group/cloud-ai"  #@param {type:"string"}

import os
os.environ["CLOUD_BACKEND"] = "unsloth"
os.environ["CLOUD_PRINT_TOKEN"] = "1"   # a notebook is yours alone; the CLI path keeps the token out of its history
os.environ["CLOUD_UNSLOTH_MODEL"] = MODEL
if TEMPLATE: os.environ["CLOUD_UNSLOTH_TEMPLATE"] = TEMPLATE
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 2 · Google Drive (skipped when already mounted, e.g. by `colab drivemount`)
need = MOUNT_DRIVE == "yes" or (MOUNT_DRIVE == "auto" and MODEL.startswith("/content/drive"))
if need and not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
print("drive:", "mounted" if os.path.ismount("/content/drive") else "not mounted")

In [ ]:
#@title 3 · Install Unsloth and fetch the router
!pip -q install unsloth
!rm -rf /content/cloud-ai && git clone --depth 1 {REPO} /content/cloud-ai
!pip -q install -r /content/cloud-ai/backend/requirements.txt

In [ ]:
#@title 4 · Start the router and the tunnel, print your VS Code settings
%run /content/cloud-ai/backend/colab_start.py

In [ ]:
#@title 5 · Wait for the model, then verify through the tunnel
import json, time, urllib.request
for _ in range(120):
    req = urllib.request.Request(os.environ["CLOUD_URL"] + "/health", headers={"Authorization": "Bearer " + os.environ["CLOUD_TOKEN"]})
    h = json.load(urllib.request.urlopen(req, timeout=30))
    print(h["status"], h["routing"])
    if h["status"] != "loading": break
    time.sleep(10)
print(json.dumps(h, indent=2))